In [1]:
import numpy as np  
import pandas as pd

from sklearn import set_config
set_config(display="text")

# 베르누이 나이브베이즈 ( 있다 없다 모델?? ㅋ )
from sklearn.naive_bayes import BernoulliNB

# 문장을 단어별 숫자 데이터로 변환
from sklearn.feature_extraction.text import CountVectorizer

from sklearn.metrics import accuracy_score

# 문제 정의

##### 베르누이 나이브베이즈 분류 모델을 사용하여 스팸 메일을 분류.

# 데이터 수집

##### 이번 실습에서는 간단한 스팸 메일 분류 실습을 위해 아래 이메일 타이틀과 스팸 여부가 있는 데이터를 사용.

In [3]:
email_list = [
                {'email title': 'free game only today', 'spam': True},
                {'email title': 'cheapest flight deal', 'spam': True},
                {'email title': 'limited time offer only today only today', 'spam': True},
                {'email title': 'today meeting schedule', 'spam': False},
                {'email title': 'your flight schedule attached', 'spam': False},
                {'email title': 'your credit card statement', 'spam': False}
             ]


df = pd.DataFrame(email_list)
df



,email title,spam
0,free game only today,True
1,cheapest flight deal,True
2,limited time offer only today only today,True
3,today meeting schedule,False
4,your flight schedule attached,False
5,your credit card statement,False


# 데이터 다듬기

##### sklearn의 베르누이 나이브베이즈 분류기는 숫자만을 다루기 때문에, True와 False를 1과 0으로 치환.

In [6]:
df['label'] = df['spam'].map({True:1, False:0})
df

,email title,spam,label
0,free game only today,True,1
1,cheapest flight deal,True,1
2,limited time offer only today only today,True,1
3,today meeting schedule,False,0
4,your flight schedule attached,False,0
5,your credit card statement,False,0


In [13]:
# 학습에 사용될 데이터와 분류값을 나눔.
df_x = df['email title']

# 정답 데이터 
df_y = df['label']

In [ ]:
cv = CountVectorizer(binary=True)
x_traincv = cv.fit_transform(df_x)

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 23 stored elements and shape (6, 17)>

In [ ]:
# 단어 목록 (컬럼 순서)
cv.get_feature_names_out()

array(['attached', 'card', 'cheapest', 'credit', 'deal', 'flight', 'free',
       'game', 'limited', 'meeting', 'offer', 'only', 'schedule',
       'statement', 'time', 'today', 'your'], dtype=object)

In [ ]:
# 단어와 컬럼 인덱스
# 단어별 컬럼 인덱스 번호 
cv.vocabulary_

{'free': 6,
 'game': 7,
 'only': 11,
 'today': 15,
 'cheapest': 2,
 'flight': 5,
 'deal': 4,
 'limited': 8,
 'time': 14,
 'offer': 10,
 'meeting': 9,
 'schedule': 12,
 'your': 16,
 'attached': 0,
 'credit': 3,
 'card': 1,
 'statement': 13}

In [24]:
encoded_input = x_traincv.toarray()
encoded_input


array([[0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0],
       [0, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 1, 0, 0, 1, 1, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0],
       [1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1],
       [0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1]])

In [32]:
cv.inverse_transform(encoded_input[[0,1,2,3,4,5]])

[array(['free', 'game', 'only', 'today'], dtype='<U9'),
 array(['cheapest', 'deal', 'flight'], dtype='<U9'),
 array(['limited', 'offer', 'only', 'time', 'today'], dtype='<U9'),
 array(['meeting', 'schedule', 'today'], dtype='<U9'),
 array(['attached', 'flight', 'schedule', 'your'], dtype='<U9'),
 array(['card', 'credit', 'statement', 'your'], dtype='<U9')]

## 베르누이 나이브베이즈 분류 모델 생성

In [34]:
# 학습 데이터로 베르누이 분류기를 학습.
bnb = BernoulliNB()

# 학습 데이터의 정답을 정수형으로 변환
y_train = df_y.astype('int')

# 학습!! 
bnb.fit(x_traincv, y_train)

BernoulliNB()

In [39]:
# 테스트 데이터 다듬기
test_email_list = [
                {'email title': 'free flight offer', 'spam': True},
                {'email title': 'hey traveler free flight deal', 'spam': True},
                {'email title': 'limited free game offer', 'spam': True},
                {'email title': 'today flight schedule', 'spam': False},
                {'email title': 'your credit card attached', 'spam': False},
                {'email title': 'free credit card offer only today', 'spam': False}
             ]


test_df = pd.DataFrame(test_email_list)
test_df['label'] = test_df['spam'].map({True:1, False:0})
test_df


,email title,spam,label
0,free flight offer,True,1
1,hey traveler free flight deal,True,1
2,limited free game offer,True,1
3,today flight schedule,False,0
4,your credit card attached,False,0
5,free credit card offer only today,False,0


In [38]:
test_x = test_df['email title']
test_y = test_df['label'].astype('int')

In [ ]:
# 이미 만든 단어 사전을 그대로 사용해 문장을 숫자로 변환 
# 앞의 모델을 테스트 하기 위해 이미 만든 단어 사전을 사용해서 사용하자 
x_testcv = cv.transform(test_x)

## 테스트

In [46]:

# 예측 할때는 데이터만 넘기고 정답은 넘기지 않는다 
predicted = bnb.predict(x_testcv)

## 정확도

In [47]:
# 정확도 확인 정답과 예측한 결과를 넣어주면 정확도가 나온다 
accuracy_score(test_y, predicted)

0.8333333333333334